# 《环境规划与管理》第八讲：US DOE TECHTEST 1.0 全量分析与敏感性分析实践 (Method 1)
**授课教师**：武汉大学 汤思琪
**授课日期**：2026年10月20日

---
## 实践内容与工程架构
本实验运行美国能源部（US DOE）**TECHTEST 1.0 全量工程分析法（Method 1: Full Analytical TEA & LCA Engine）**，并结合工业余热回收与先进催化剂案例进行参数敏感性（Sensitivity Analysis）分析。

核心学习内容：
1. **工程明细级建模**：设备折旧（CapEx 直线折旧 + 5%维护）、Harry Silla 2003 工厂人工建模、原材料流（MFI）与公用能源（EIA/GWP）核算；
2. **基准技术非等量替代比（Equivalence Ratio）**处理；
3. **电价波动与电网深度脱碳情景**（2022 Baseline vs. Net Zero）对技术经济性与减碳潜力的动态敏感度。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from pytechtest import TechTestFullEngine, TechTestReporter, GridScenario
print("✅ US DOE TECHTEST 1.0 Full Analytical Engine 已就绪！")

## 1. 运行工业余热回收基准案例 (Waste Heat Recovery)

In [ ]:
case1_path = Path.cwd().parent / "examples" / "waste_heat_recovery.yaml"
engine1 = TechTestFullEngine.from_yaml(case1_path)
res1 = engine1.run()

reporter = TechTestReporter()
reporter.print_terminal_dashboard(res1)

## 2. 运行先进催化剂非等量替代案例 (Manufacturilium Catalyst)
注意：该案例 10 kg 新型催化剂等效替代传统催化剂 13.5 kg，且包含 -/基准体积的副产物回收抵扣。

In [ ]:
case2_path = Path.cwd().parent / "examples" / "manufacturilium_catalyst.yaml"
engine2 = TechTestFullEngine.from_yaml(case2_path)
res2 = engine2.run()

reporter.print_terminal_dashboard(res2)

## 3. 电网清洁化转型敏感性分析 (Grid Mix Decarbonization Sweep)
对比 2022 基准电网、温和清洁情景（Moderate）与零碳电网（Net Zero）对单位产品碳足迹的影响：

In [ ]:
scenarios = [GridScenario.BASELINE_2022, GridScenario.MODERATE, GridScenario.NET_ZERO]
scen_names = ["2022现状电网", "温和脱碳情景", "Net Zero零碳电网"]
emissions_new = []
emissions_bench = []

for scen in scenarios:
    inp = engine2.config.model_copy(deep=True)
    inp.grid_scenario = scen
    eng = TechTestFullEngine(inp)
    r = eng.run()
    emissions_new.append(r.new_technology.total_emissions_per_reference_volume_lb)
    emissions_bench.append(r.commercial_benchmark.total_emissions_per_reference_volume_lb)

plt.figure(figsize=(9, 5))
x = np.arange(len(scenarios))
width = 0.35
plt.bar(x - width/2, emissions_bench, width, label="传统基准技术", color="#d9534f")
plt.bar(x + width/2, emissions_new, width, label="绿色新技术", color="#5cb85c")
plt.xticks(x, scen_names, fontsize=11)
plt.ylabel("摇篮到大门碳排放 (lb CO2e / 10kg催化剂)", fontsize=11)
plt.title("电网脱碳情景对技术碳减排效益的动态影响", fontsize=13, fontweight="bold")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 4. 关键能源价格扰动敏感性扫描 (Electricity Price Sensitivity)
考察工业电价从 0.05 $/kWh 波动至 0.25 $/kWh 时，新技术相对于基准技术的单耗成本节约优势：

In [ ]:
elec_prices = np.linspace(10.0, 50.0, 20)  # $/MMBtu (~0.034 to 0.17 $/kWh)
cost_savings_list = []

for ep in elec_prices:
    inp = engine1.config.model_copy(deep=True)
    # 修改电力单价
    for item in inp.new_technology.energy_items:
        if "electricity" in item.energy_source.lower():
            item.unit_price = ep
    for item in inp.commercial_benchmark.energy_items:
        if "electricity" in item.energy_source.lower():
            item.unit_price = ep
    eng = TechTestFullEngine(inp)
    r = eng.run()
    cost_savings_list.append(r.summary.cost_savings_per_ref_volume)

plt.figure(figsize=(9, 4.5))
plt.plot(elec_prices, cost_savings_list, marker="o", linewidth=2, color="#0275d8")
plt.xlabel("工业电价 ($/MMBtu)", fontsize=11)
plt.ylabel("单位基准产品制造成本节约 ($/ref)", fontsize=11)
plt.title("工业余热回收技术：电价敏感性曲线", fontsize=13, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()